# B-TRUST Phase 73 — transformers at k = 8, item-level predictions, and the LLM (Colab, GPU)
**Preregistered here, before running** (written 2026-11-19, after Phase 70 results were seen: H9a/H9c failed at k = 5; in Phase 70 RoBERTa trained on
SCAR-E scored 33/70 on YouLeQD). Record the run date and this notebook's SHA-256 with the results.

- **H12a** DistilBERT (`distilbert-base-uncased`): Spearman ρ(G0 lift, mean transfer T) > 0 over the k = 8 corpora of Phase 71; exact one-sided
  permutation test over all 8! orderings, α = 0.05.
- **H12b** same under `roberta-base`.
- **H12c** for each model, every G0-PASS corpus has higher T than every G0-FAIL corpus (Phase 70 observation, now tested at k = 8; descriptive pass/fail).
- **YouLeQD item-level predictions are saved for descriptive use only.** The Phase 70 RoBERTa/SCAR-E observation was made on these same 70 items,
  so it cannot be confirmed here; its confirmatory test is reserved for the new annotated learner-question set.
- **H10 (unchanged from Phase 70)** zero-shot LLM G6 vote loss on YouLeQD 70 below the best constant (0.738); paired video-cluster bootstrap 10,000,
  seed 70; supported iff 95% CI upper bound < 0.

Fixed settings as Phase 70: max_len 64, batch 16, lr 2e-5, 3 epochs, weight decay 0.01, seed 70, linear schedule. Corpora, targets, leakage rule,
G0 thresholds and the SCAR-I/SCAR-E exclusion exactly as Phase 71.

## 1. Setup — upload `BTRUST_Frozen_Supporting_Files_2026-11-07.zip` and `BTRUST_v2.4_*.zip` when prompted (GPU runtime)

In [ ]:
!pip -q install transformers==4.46.3 accelerate anthropic scikit-learn pandas openpyxl
import os, zipfile, glob, subprocess, sys
from google.colab import files
up = files.upload()
for z in up: zipfile.ZipFile(z).extractall('/content/in')
os.makedirs('/content/ex', exist_ok=True)
for z in glob.glob('/content/in/**/original_phase_archives/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall(f'/content/ex/{os.path.basename(z)[:-4]}')
os.environ['BTRUST_EX'] = '/content/ex'
code_dir = glob.glob('/content/in/**/v2.0_phases65-68/code', recursive=True)[0]
FARAJI = glob.glob('/content/in/**/data_faraji_MIT', recursive=True)[0]
os.chdir(code_dir); subprocess.run(['sh','fetch_and_prepare.sh'], check=True); sys.path.insert(0, code_dir)

## 2. Corpora — identical to Phase 71

In [ ]:
import numpy as np, pandas as pd, itertools, torch, random
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import corpora as C
REV = {"remember":1,"understand":2,"apply":3,"analyze":4,"evaluate":5,"create":6}
def faraji(f):
    d = pd.read_csv(f'{FARAJI}/{f}'); d = pd.DataFrame({"text": d.QUESTION, "y": d["BT LEVEL"].str.strip().str.lower().map(REV)})
    d = d[d.text.map(lambda s: isinstance(s, str) and s.strip() != "")].copy(); d["n"] = d.text.map(C.norm)
    return d.drop_duplicates("n").reset_index(drop=True)
corp = {"EDUB": C.edub(), "YAHYA": faraji("yahya-2012_revised.csv"), "SANGO": faraji("Sangodiah_2017_415_revised.csv"),
        "MOHOM": faraji("Mohammed-and-Omar_2020_126_revised.csv"), "GANI": faraji("Gani_2023_collected-dataset-1200_revised.csv"),
        "SCAR-I": C.scar_i(), "SCAR-E": C.scar_e(), "EDUQG": C.eduqg()}
yq, V = C.youleqd(); lk = C.lak(); TG = ["YAHYA","SANGO","MOHOM","GANI","SCAR-E","EDUQG"]
expected = {'EDUB':2550,'YAHYA':599,'SANGO':413,'MOHOM':126,'GANI':1185,'SCAR-I':2550,'SCAR-E':1839,'EDUQG':903}
assert {k: len(v) for k, v in corp.items()} == expected, {k: len(v) for k, v in corp.items()}
assert len(yq) == 70 and len(lk) == 40

## 3. Fine-tuning helper (unchanged from Phase 70)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def seed_all(s=70): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def fit_predict(name, Xtr, ytr, Xte_list):
    seed_all(); tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=6).to(dev)
    enc = tok(list(Xtr), truncation=True, max_length=64, padding=True, return_tensors='pt'); y = torch.tensor(np.asarray(ytr) - 1)
    idx = np.arange(len(y)); opt = torch.optim.AdamW(m.parameters(), lr=2e-5, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, 0, 3 * int(np.ceil(len(y) / 16))); m.train(); rg = np.random.default_rng(70)
    for ep in range(3):
        rg.shuffle(idx)
        for i in range(0, len(idx), 16):
            b = idx[i:i+16]
            out = m(input_ids=enc['input_ids'][b].to(dev), attention_mask=enc['attention_mask'][b].to(dev), labels=y[b].to(dev))
            out.loss.backward(); opt.step(); sch.step(); opt.zero_grad()
    m.eval(); preds = []
    for X in Xte_list:
        P = []
        for i in range(0, len(X), 64):
            e = tok(list(X[i:i+64]), truncation=True, max_length=64, padding=True, return_tensors='pt').to(dev)
            with torch.no_grad(): P.append(m(**e).logits.argmax(-1).cpu().numpy() + 1)
        preds.append(np.concatenate(P) if P else np.array([], int))
    return preds

## 4. Gates and transfer, k = 8 (H12a–H12c); saves YouLeQD item-level predictions

In [ ]:
def lookup_cv(key, y):
    p = np.zeros(len(y), int); maj = pd.Series(y).mode()[0]
    for a, b in StratifiedKFold(5, shuffle=True, random_state=65).split(key, y):
        t = pd.DataFrame({'k': key[a], 'y': y[a]}).groupby('k').y.agg(lambda s: s.mode()[0]); p[b] = pd.Series(key[b]).map(t).fillna(maj).astype(int).to_numpy()
    return p
items = []
for name in ["distilbert-base-uncased", "roberta-base"]:
    rows = []
    for c, d in corp.items():
        X = d.text.to_numpy(); y = d.y.to_numpy(); p = np.zeros(len(y), int)
        for a, b in StratifiedKFold(5, shuffle=True, random_state=65).split(X, y): p[b] = fit_predict(name, X[a], y[a], [X[b]])[0]
        acc = (p == y).mean(); maj = pd.Series(y).value_counts(normalize=True).iloc[0]
        meta = (lookup_cv(d.meta.to_numpy(), y) == y).mean() if 'meta' in d else np.nan
        lift = acc - np.nanmax([maj, meta]); k = cohen_kappa_score(y, p)
        gate = "FAIL" if (lift < .05 or k < .20) else ("PASS" if (lift >= .15 and k >= .40) else "CAUTION")
        ts = set(d.n); tests, names = [], []
        for tn in TG:
            if tn == c or {c, tn} == {"SCAR-I", "SCAR-E"}: continue
            t = corp[tn]; keep = ~t.n.isin(ts); tests.append(t.text[keep].to_numpy()); names.append((tn, t.y[keep].to_numpy()))
        P = fit_predict(name, X, y, tests + [yq.text.to_numpy(), lk.text.to_numpy()])
        accs = {tn: (pp == yt).mean() for (tn, yt), pp in zip(names, P[:len(names)])}
        accs["YLQ70"] = (P[-2] == yq.y.to_numpy()).mean(); accs["LAK40"] = np.mean([a in s for a, s in zip(P[-1], lk.refs)])
        items.append(pd.DataFrame({"model": name, "train": c, "row": np.arange(70), "vid": yq.vid.values, "y": yq.y.values, "pred": P[-2]}))
        rows.append(dict(model=name, train=c, n=len(y), cv_acc=acc, G0_lift=lift, kappa_cv=k, G0=gate, **accs, T=np.mean(list(accs.values()))))
        print(name, c, gate, round(lift, 3), round(rows[-1]["T"], 3))
    R = pd.DataFrame(rows); R.to_csv(f"p73_k8_{name.split('-')[0]}.csv", index=False)
    x, yy = R.G0_lift.to_numpy(), R["T"].to_numpy(); obs = spearmanr(x, yy).statistic
    pv = np.mean([spearmanr(x, np.array(q)).statistic >= obs - 1e-12 for q in itertools.permutations(yy)])
    sep = R[R.G0 == "PASS"]["T"].min() > R[R.G0 == "FAIL"]["T"].max()
    print(f"{name}: H12 rho={obs:.3f}, exact p={pv:.4f}, supported={pv < 0.05}; H12c PASS>FAIL separation={sep}")
pd.concat(items).to_csv("p73_youleqd_item_predictions.csv", index=False)

## 5. Zero-shot LLM (H10, unchanged). Enter your own API key; about 2,000 short calls.

In [ ]:
import anthropic, getpass, re, time
client = anthropic.Anthropic(api_key=getpass.getpass('Anthropic API key: '))
MODEL = 'claude-sonnet-5-5'   # record the exact model string used
PROMPT = ('Classify the cognitive level required to answer the question below using the original Bloom taxonomy. '
          'Answer with one word only: Knowledge, Comprehension, Application, Analysis, Evaluation, or Synthesis.\n\nQuestion: ')
MAP = {"knowledge":1,"comprehension":2,"application":3,"analysis":4,"evaluation":5,"synthesis":6}
def llm(t):
    for _ in range(5):
        try:
            r = client.messages.create(model=MODEL, max_tokens=5, temperature=0, messages=[{"role":"user","content":PROMPT+t}])
            return MAP.get(re.sub(r'[^a-z]', '', r.content[0].text.lower()), 0)
        except Exception: time.sleep(5)
    return 0
rg = np.random.default_rng(70); out = {}
for k in ["YAHYA","SANGO","MOHOM","GANI","SCAR-E","EDUQG"]:
    d = corp[k]; d = d.iloc[rg.choice(len(d), min(500, len(d)), replace=False)]
    out[k] = (np.array([llm(t) for t in d.text]) == d.y.to_numpy()).mean(); print(k, out[k])
pq = np.array([llm(t) for t in yq.text]); out["YLQ70"] = (pq == yq.y.to_numpy()).mean()
pl = np.array([llm(t) for t in lk.text]); out["LAK40"] = np.mean([a in s for a, s in zip(pl, lk.refs)])
vl = np.nanmean(np.where(np.isnan(V), 1.0, (V != pq[:, None]).astype(float)), 1)
const = np.nanmean(np.where(np.isnan(V), 1.0, (V != 2).astype(float)), 1)
d = vl - const; g = pd.Series(d).groupby(yq.vid.values); s = g.sum().to_numpy(); c = g.size().to_numpy(); rb = np.random.default_rng(70)
bs = [s[i].sum() / c[i].sum() for i in (rb.integers(0, len(s), len(s)) for _ in range(10000))]
print(out); print(f"H10: LLM {vl.mean():.4f} vs constant {const.mean():.4f}; diff {d.mean():+.4f}, CI [{np.percentile(bs,2.5):+.4f}, {np.percentile(bs,97.5):+.4f}]; supported={np.percentile(bs,97.5) < 0}")
pd.DataFrame([dict(model=MODEL, **out, ylq_voteloss=vl.mean(), h10_ci_hi=np.percentile(bs, 97.5))]).to_csv("p73_llm.csv", index=False)
pd.DataFrame({"row": np.arange(70), "vid": yq.vid.values, "y": yq.y.values, "pred": pq}).to_csv("p73_llm_youleqd_items.csv", index=False)

## 6. Download all `p73_*.csv` files.

In [ ]:
import glob; [files.download(f) for f in glob.glob('p73_*.csv')]